In [1]:
# Outside a managed BQ notebook, authenticate first:
# from google.colab import auth; auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = None  # e.g. "my-gcp-project"; None => environment default
DATASET    = "emergency_calls"
LOCATION   = "US"

client = bigquery.Client(project=PROJECT_ID) if PROJECT_ID else bigquery.Client()
PROJECT_ID = client.project
print("Using project:", PROJECT_ID)

RAW_TABLE = f"{PROJECT_ID}.{DATASET}.calls_raw"
MODEL     = f"{PROJECT_ID}.{DATASET}.response_time_model"

def run(sql):
    job = client.query(sql)
    job.result()
    return job


Using project: qwiklabs-gcp-01-9ed82c2a1f67


In [2]:
run(f"""
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}`
OPTIONS(location="{LOCATION}")
""")
print("Dataset ready:", DATASET)


Dataset ready: emergency_calls


In [13]:
## 2. Import the CSV into BigQuery
run(f"""
LOAD DATA OVERWRITE `{RAW_TABLE}`
FROM FILES (
  format = 'CSV',
  uris   = ['gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv'],
  skip_leading_rows = 1,
  field_delimiter = ',',
  max_bad_records = 10
)
""")
print("Loaded:", RAW_TABLE)


Loaded: qwiklabs-gcp-01-9ed82c2a1f67.emergency_calls.calls_raw


In [4]:
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 10").to_dataframe()

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


In [6]:
## 3. Study the data

# Row count and response_time distribution
client.query(f"""
SELECT
  COUNT(*)            AS row_count,
  MIN(response_time)  AS min_rt,
  AVG(response_time)  AS avg_rt,
  MAX(response_time)  AS max_rt,
  STDDEV(response_time) AS std_rt
FROM `{RAW_TABLE}`
""").to_dataframe()


,row_count,min_rt,avg_rt,max_rt,std_rt
0,50000,2.01,17.446134,36.55,5.296881


In [7]:
## 4. Create the model with BigQuery ML

# Categorical levels — confirms what the model will one-hot encode
client.query(f"""
SELECT 'call_type' AS col, call_type AS value, COUNT(*) n FROM `{RAW_TABLE}` GROUP BY value
UNION ALL SELECT 'traffic_level', traffic_level, COUNT(*) FROM `{RAW_TABLE}` GROUP BY traffic_level
UNION ALL SELECT 'weather_condition', weather_condition, COUNT(*) FROM `{RAW_TABLE}` GROUP BY weather_condition
ORDER BY col, n DESC
""").to_dataframe()


,col,value,n
0,call_type,Fire,12585
1,call_type,Police,12536
2,call_type,Medical,12478
3,call_type,Rescue,12401
4,traffic_level,High,16744
5,traffic_level,Medium,16655
6,traffic_level,Low,16601
7,weather_condition,Sunny,12633
8,weather_condition,Snowy,12558
9,weather_condition,Rainy,12429


In [8]:
## 5. Evaluate with `ML.EVALUATE`

# Average response time by traffic level — a quick signal check
client.query(f"""
SELECT traffic_level, ROUND(AVG(response_time),2) AS avg_response_time, COUNT(*) AS n
FROM `{RAW_TABLE}`
GROUP BY traffic_level
ORDER BY avg_response_time DESC
""").to_dataframe()


,traffic_level,avg_response_time,n
0,High,20.09,16744
1,Medium,17.14,16655
2,Low,15.08,16601


In [9]:
run(f"""
CREATE OR REPLACE MODEL `{MODEL}`
OPTIONS (
  model_type        = 'LINEAR_REG',
  input_label_cols  = ['response_time'],
  data_split_method = 'AUTO_SPLIT'
) AS
SELECT
  call_type,
  location,
  weather_condition,
  day_of_week,
  time_of_day,
  traffic_level,
  distance_to_station,
  units_available,
  response_time
FROM `{RAW_TABLE}`
WHERE response_time IS NOT NULL
""")
print("Model trained:", MODEL)


Model trained: qwiklabs-gcp-01-9ed82c2a1f67.emergency_calls.response_time_model


In [10]:
eval_df = client.query(f"""
SELECT * FROM ML.EVALUATE(MODEL `{MODEL}`)
""").to_dataframe()
eval_df


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.761934,4.827846,0.015117,1.501836,0.831417,0.83146


In [11]:
# Feature weights — which inputs move the prediction most
client.query(f"""
SELECT processed_input, weight
FROM ML.WEIGHTS(MODEL `{MODEL}`)
WHERE weight IS NOT NULL
ORDER BY ABS(weight) DESC
LIMIT 20
""").to_dataframe()


,processed_input,weight
0,__INTERCEPT__,192.055265
1,distance_to_station,0.450567
2,units_available,-0.140156
3,time_of_day,-0.000166


In [12]:
## 6. Predict on synthetic data with `ML.PREDICT`

pred_df = client.query(f"""
SELECT *
FROM ML.PREDICT(MODEL `{MODEL}`,
  (
    SELECT 'Medical' AS call_type, 'Downtown' AS location, 'Snowy' AS weather_condition,
           'Monday' AS day_of_week, 8 AS time_of_day, 'High' AS traffic_level,
           12.5 AS distance_to_station, 2 AS units_available
    UNION ALL
    SELECT 'Fire', 'Greenfield', 'Sunny', 'Saturday', 15, 'Low', 4.2, 9
    UNION ALL
    SELECT 'Police', 'Uptown', 'Rainy', 'Friday', 22, 'Medium', 9.0, 4
  ))
""").to_dataframe()
pred_df


,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,23.559482,Medical,Downtown,Snowy,Monday,8,High,12.5,2
1,7.347054,Fire,Greenfield,Sunny,Saturday,15,Low,4.2,9
2,13.160037,Police,Uptown,Rainy,Friday,22,Medium,9.0,4
